In [ ]:
import os
import re
import glob
from collections import Counter

import requests
from bs4 import BeautifulSoup
import fitz
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation, PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

# CONFIGURATION 
gaza_id = 1081 

gaza_stopwords = [
    'gaza', 'israel', 'israeli', 'palestine', 'palestinian', 'palestinians', 
    'forces', 'including', 'that', 'with', 'from', 'their', 
    'report', 'pursuant', 'resolution', 'submitted', 'office', 
    'united', 'nations', 'human', 'rights', 'council', 'present', 
    'provides', 'providing', 'update', 'examines', 'prepared', 'ohchr',
    'state', 'general', 'assembly', 'october', 'documented', 'information',
    'according', 'also', 'would', 'could', 'may', 'might', 'must', 'shall'
]

data = {
    "Gaza": {"links": [], "texts": [], "custom_stopwords": gaza_stopwords}
}

# SCRAPING
base_url = "https://www.ohchr.org/en/documents-listing"
headers = { "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36" }


print(f"\n🌍 Scraping for Gaza...\n")

params = {
    f"field_geolocation_target_id[{gaza_id}]": gaza_id,
    "field_content_category_target_id[180]": 180,
    "field_content_category_target_id[182]": 182,
    "field_entity_target_id[1349]": 1349,
    "field_entity_target_id[1350]": 1350,
    "view_mode": "document-list",
    "sort_bef_combine": "field_published_date_value_DESC"
}

page = 0
while True:
    params["page"] = page
    print(f"📄 Reading page {page + 1} -> Gaza...")
    response = requests.get(base_url, params=params, headers=headers)
    if response.status_code != 200:
        print("⚠️ End of pagination or HTTP error:", response.status_code)
        break

    soup = BeautifulSoup(response.text, "html.parser")
    cards = soup.find_all("a", class_="card-2__link", href=True)
    page_links = []

    for a in cards:
        text = a.get_text(strip=True)
        href = a["href"]
        if "A/" in text or "a/" in href.lower():
            full_url = "https://www.ohchr.org" + href
            page_links.append(full_url)

    if not page_links:
        print("❌ No links found — probably end of pagination.")
        break

    print(f"✨ {len(page_links)} link(s) found on page {page + 1}.")
    data["Gaza"]["links"].extend(page_links)
    page += 1

# DOWNLOAD - this will create a folder and download the documents there
# we have also included the documents in a folder Palestine documents in case something goes wrong with the downloading 
# In that case, skip to line 'folder = f"docs/gaza"' and change the path name to PalestineDocuments
def extract_code(link):
    match = re.search(r"ahrc(\d{4})(?:-|$)", link, re.IGNORECASE)
    if match:
        digits = match.group(1)
        return f"A/HRC/{digits[:2]}/{digits[2:]}"
    
    match = re.search(r"ahrc(\d{5})(?:-|$)", link, re.IGNORECASE)
    if match:
        digits = match.group(1)
        return f"A/HRC/{digits[:2]}/{digits[2:]}"
    
    match = re.search(r"ahrc(\d+)(crp|wp|res|add)(\d+)?", link, re.IGNORECASE)
    if match:
        session = match.group(1)
        suffix_type = match.group(2).upper()
        suffix_num = match.group(3) if match.group(3) else ""
        return f"A/HRC/{session}/{suffix_type}.{suffix_num}" if suffix_num else f"A/HRC/{session}/{suffix_type}"
    
    match = re.search(r"/a(\d{5})(?:-|$)", link, re.IGNORECASE)
    if match:
        digits = match.group(1)
        return f"A/{digits[:2]}/{digits[2:]}"
    
    return None

os.makedirs(f"docs/gaza", exist_ok=True)
codes = []
for link in data["Gaza"]["links"]:
    code = extract_code(link)
    codes.append(code)

    if not code:
        print(f"⚠️ No code found in: {link}")

successful = len([c for c in codes if c])
print(f"\n🌍 Gaza: {successful}/{len(codes)} code(s) extracted from link(s) successfully, downloading...")

for code in codes:
    if not code:
        continue

    pdf_url = f"https://documents.un.org/api/symbol/access?s={code}&l=en&t=pdf"

    try:
        response = requests.get(pdf_url)
        response.raise_for_status()

        filename = code.replace("/", "_") + ".pdf"
        filepath = os.path.join("docs", 'gaza', filename)

        with open(filepath, "wb") as f:
            f.write(response.content)

        print(f"✅ PDF downloaded: {filename}")

    except requests.HTTPError as e:
        print(f"⚠️ Unable to download {pdf_url}: {e}")

# CLEANING
def extract_metadata_from_first_page(page):
    metadata = {"symbol": None, "date": None, "summary": None}
    
    m = re.search(r'\b[ASE]\/[A-Z0-9][A-Z0-9\/\.\-]*\b', page)
    if m:
        metadata["symbol"] = m.group(0)
    
    dates = re.findall(r'\b\d{1,2}\s+[A-Z][a-z]+\s+\d{4}\b', page)
    if dates:
        metadata["date"] = dates[-1]

    summary_start = re.search(r'\bSummary\b', page, flags=re.I)
    if summary_start:
        text_after_summary = page[summary_start.end():]
        summary_clean = re.split(r'(?:\n\s*){3,}', text_after_summary)[0]
        summary_clean = re.sub(r'\s+', ' ', summary_clean).strip()
        metadata["summary"] = summary_clean

    return metadata

def clean_page(page):
    page = re.split(r'(?:\n\s*){4,}', page)[0]
    page = re.split(r'_{5,}', page)[0]
    page = page.replace("\n", " ")
    page = re.sub(r'\b[AES]\/[A-Z0-9][A-Z0-9\/\.\-]*\b', '', page)
    page = re.sub(r'\d+', '', page)
    page = re.sub(r'[^A-Za-zÀ-ÖØ-öø-ÿ\s.!?]', '', page)
    page = re.sub(r'\bGE\b', '', page)
    page = re.sub(r'\s+', ' ', page).strip()

    return page

print(f"\n📂 Loading and cleaning documents...")

data['Gaza']["texts"] = []
folder = f"docs/gaza" 

pdf_files = glob.glob(f"{folder}/*.pdf")

if not pdf_files:
    print("⚠️ No PDF found in this folder.")

for file in pdf_files:
    doc = fitz.open(file)
    pages_text = [p.get_text() for p in doc]
    doc.close()

    processed_meta = extract_metadata_from_first_page(pages_text[0])
    symbol  = processed_meta["symbol"]  or "UNKNOWN_SYMBOL"
    date    = processed_meta["date"]    or "UNKNOWN_DATE"
    summary = processed_meta["summary"] or "NO_SUMMARY_AVAILABLE"

    meta = f"{symbol} - {date} - {summary}"
    text_cleaned = " ".join(clean_page(p) for p in pages_text[1:])
    data["Gaza"]["texts"].append([meta, text_cleaned])

#  PHASE 4 : NLP PRE-PROCESSING 
lemmatizer = WordNetLemmatizer()

def preprocess_text(text, custom_stopwords, format_mode):
    stop_words = set(stopwords.words('english'))
    stop_words.update(custom_stopwords)
    
    tokens = word_tokenize(text.lower())
    if format_mode:
        tokens = [lemmatizer.lemmatize(t) for t in tokens if t.isalpha()]
        tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
        tagged = nltk.pos_tag(tokens)
        pos_tags=['VB', 'JJ', 'RB', 'NN']
        filtered = [word for word, tag in tagged if any(tag.startswith(p) for p in pos_tags)]
        return filtered
    return tokens

print("\n🔧 NLP Pre-processing...")
new_texts = []
for meta, text_cleaned in data["Gaza"]["texts"]:
    processed = preprocess_text(text_cleaned, data["Gaza"]["custom_stopwords"], 1)
    text_final = " ".join(processed)
    new_texts.append((meta, text_cleaned, text_final))
data["Gaza"]["texts"] = new_texts

#  PHASE 5 : WORD FREQUENCY 
print("\n📊 Word frequency analysis...")
results = {}

texts = [t[2] for t in data["Gaza"]["texts"]]
all_words = " ".join(texts).split()
word_freq = Counter(all_words).most_common(20)

results["Gaza"] = {"word_frequency": word_freq}

#  PHASE 6 : TF-IDF 
print("\n📊 TF-IDF calculation...")

vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
tfidf_matrix = vectorizer.fit_transform(texts)
tfidf = sorted(zip(vectorizer.get_feature_names_out(), tfidf_matrix.sum(axis=0).A1), key=lambda x: x[1], reverse=True)[:20]

new_texts = []
for (meta, text_cleaned, text_final) in data["Gaza"]["texts"]:
    scores = vectorizer.transform([text_final]).toarray()[0]
    tf_idf = sorted(zip(vectorizer.get_feature_names_out(), scores), key=lambda x: x[1], reverse=True)[:15]
    new_texts.append((meta, text_cleaned, text_final, tf_idf))

data["Gaza"]["texts"] = new_texts
results["Gaza"]["tfidf_text"] = tfidf

#  PHASE 7 : SENTIMENT ANALYSIS 
print("\n📊 Sentiment analysis...")

sent = TextBlob(' '.join(texts)).sentiment._asdict()
results["Gaza"]["sentiment_analysis"] = sent

new_texts = []
for (meta, text_cleaned, text_final, tf_idf) in data["Gaza"]["texts"]:
    sent = TextBlob(text_final).sentiment._asdict()
    new_texts.append((meta, text_cleaned, text_final, tf_idf, sent))

data["Gaza"]["texts"] = new_texts

#  PHASE 8 : TOPIC MODELING (LDA) 
print("\n🎯 Topic Modeling (LDA)...")

if len(texts) < 2:
    print(f"⚠️ Not enough documents for LDA in Gaza")

vectorizer_lda = CountVectorizer(max_features=200, stop_words='english', min_df=2)
counts = vectorizer_lda.fit_transform(texts)

n_topics = min(4, len(texts))
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, max_iter=50)
lda.fit(counts)

feature_names = vectorizer_lda.get_feature_names_out()

lda_topics = []
for topic_idx, topic in enumerate(lda.components_):
    top_indices = topic.argsort()[-20:][::-1]
    top_words = [feature_names[i] for i in top_indices]
    lda_topics.append((topic_idx, top_words))

doc_topic_dist = lda.transform(counts)

# Extract dominant topic and year for each document
dominant_topics = []
for idx, (meta, _, _, _, _) in enumerate(data["Gaza"]["texts"]):
    dominant_topic = np.argmax(doc_topic_dist[idx]) + 1

    # Extract year from metadata
    year_match = re.search(r'\b(202[0-9])\b', meta)
    year = int(year_match.group(1)) if year_match else None

    dominant_topics.append((dominant_topic, year))

results["Gaza"]["lda"] = {
    "topics": lda_topics,
    "distrib_by_doc": doc_topic_dist.tolist(),
    "dominant_topics": dominant_topics
}

#  PHASE 9 : CLUSTERING (KMeans + Silhouette) 
print("\n🔍 Clustering...")

if len(texts) < 3:
    print(f"⚠️ Not enough documents for clustering in Gaza")

vectorizer_clust = TfidfVectorizer(max_df=0.90, min_df=2, stop_words='english', max_features=300)
X = vectorizer_clust.fit_transform(texts)
vocab = vectorizer_clust.get_feature_names_out()

wcss = []
silhouette_scores = []

for k in range(2, min(10, len(texts)) + 1):
    kmeans = KMeans(n_clusters=k, init='k-means++', max_iter=300, random_state=42)
    labels = kmeans.fit_predict(X)
    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X, labels))

# best_k = range(2, min(10, len(texts)) + 1)[np.argmax(silhouette_scores)] # based on silhouette scores
best_k = 4 # we decided 4 would be best for us 
kmeans = KMeans(n_clusters=best_k, init='k-means++', max_iter=300, random_state=42)
labels = kmeans.fit_predict(X)

# Get top words for each cluster
cluster_centers = kmeans.cluster_centers_
cluster_words = []
for i in range(best_k):
    top_indices = cluster_centers[i].argsort()[-50:][::-1]
    top_words = [vocab[idx] for idx in top_indices]
    cluster_words.append(top_words)
    
results["Gaza"]["clustering"] = {
    "best_k": best_k,
    "silhouette_scores": silhouette_scores,
    "wcss": wcss,
    "labels": labels.tolist(),
    "cluster_words": cluster_words
}

new_texts = []
for (meta, text_cleaned, text_final, tfidf, sent), cluster_id in zip(data["Gaza"]["texts"], labels):
    new_texts.append((meta, text_cleaned, text_final, tfidf, sent, cluster_id))
data["Gaza"]["texts"] = new_texts

# PCA for visualization
if X.shape[0] >= 2:
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X.toarray())
    results["Gaza"]["clustering"]["pca_coords"] = X_pca.tolist()

if X.shape[0] >= 2:
    pca = PCA(n_components=3, random_state=42)
    X_pca3 = pca.fit_transform(X.toarray())
    results["Gaza"]["clustering"]["pca_coords_3"] = X_pca3

#  PHASE 10 : VISUALIZATIONS 
print("\n📊 Generating visualizations...\n")

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

os.makedirs("docs/visualisation", exist_ok=True)

    
# WORD FREQUENCY
fig, ax = plt.subplots(figsize=(14, 8))
word_freq = results["Gaza"]["word_frequency"]
words = [item[0] for item in word_freq]
counts = [item[1] for item in word_freq]
bars = ax.barh(words, counts, color='#FFB6C1')
ax.invert_yaxis()
ax.set_xlabel('Count', fontsize=12, fontweight='bold')
ax.set_title(f'Top 20 Most Frequent Words - Gaza', fontsize=16, fontweight='bold', pad=20)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f"docs/visualisation/gaza_word_frequency.png", dpi=300, bbox_inches='tight')
plt.close()

# WORD CLOUD
fig, ax = plt.subplots(figsize=(16, 10))
all_text = ' '.join([t[2] for t in data["Gaza"]["texts"]])
wordcloud = WordCloud(width=1600, height=1000, background_color='white', 
                     colormap='Blues', max_words=100, relative_scaling=0.5, 
                     min_font_size=10).generate(all_text)
ax.imshow(wordcloud, interpolation='bilinear')
ax.axis('off')
ax.set_title(f'Word Cloud - Gaza', fontsize=20, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f"docs/visualisation/gaza_wordcloud.png", dpi=300, bbox_inches='tight')
plt.close()

# TF-IDF TOP WORDS PER DOCUMENT
fig, ax = plt.subplots(figsize=(12, max(8, len(data["Gaza"]["texts"]) * 0.4)))
doc_labels = [f"{i}" for i in range(len(data["Gaza"]["texts"]))]
doc_words = [", ".join([w for w, s in t[3][:15]]) for t in data["Gaza"]["texts"]]

y_pos = np.arange(len(doc_labels))
ax.set_yticks(y_pos)
ax.set_yticklabels(doc_labels)
ax.set_xlabel('Top TF-IDF Words', fontsize=12, fontweight='bold')
ax.set_title(f'Top 15 TF-IDF Words per Document - Gaza', fontsize=14, fontweight='bold')

for i, words in enumerate(doc_words):
    ax.text(0.05, i, words[:100] + "...", va='center', fontsize=8)

ax.set_xlim(0, 1)
ax.set_ylim(-0.5, len(doc_labels) - 0.5)
plt.tight_layout()
plt.savefig(f"docs/visualisation/gaza_tfidf_per_doc.png", dpi=300, bbox_inches='tight')
plt.close()

# SENTIMENT ANALYSIS TABLE
fig, ax = plt.subplots(figsize=(16, max(8, len(data["Gaza"]["texts"]) * 0.3)))
ax.axis('tight')
ax.axis('off')

table_data = []
for idx, (meta, _, _, _, sent, _) in enumerate(data["Gaza"]["texts"]):
    title = meta.split(" - ")[0] if " - " in meta else f"Doc {idx}"
    table_data.append([
        idx,
        title[:50],
        f"{sent['polarity']:.6f}",
        f"{sent['subjectivity']:.6f}"
    ])

table = ax.table(cellText=table_data,
                colLabels=['#', 'Title', 'Polarity', 'Subjectivity'],
                cellLoc='left',
                loc='center',
                colWidths=[0.05, 0.6, 0.15, 0.15])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)

plt.title(f'Sentiment Analysis per Document - Gaza', fontsize=14, fontweight='bold', pad=20)
plt.savefig(f"docs/visualisation/gaza_sentiment_table.png", dpi=300, bbox_inches='tight')
plt.close()

# LDA TOPICS - DOMINANT TOPICS COUNT
if "lda" in results["Gaza"]:
    dominant_topics = [t[0] for t in results["Gaza"]["lda"]["dominant_topics"]]
    topic_counts = Counter(dominant_topics)

    fig, ax = plt.subplots(figsize=(10, 6))
    topics = sorted(topic_counts.keys())
    counts = [topic_counts[t] for t in topics]
    ax.bar(topics, counts, color='lightblue', edgecolor='black')
    ax.set_xlabel('Dominant Topic', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Documents', fontsize=12, fontweight='bold')
    ax.set_title(f'Counts of Dominant Topics - Gaza', fontsize=14, fontweight='bold')
    ax.set_xticks(topics)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"docs/visualisation/gaza_dominant_topics_count.png", dpi=300, bbox_inches='tight')
    plt.close()

    # DOMINANT TOPICS PER YEAR
    year_topic_data = {}
    for topic, year in results["Gaza"]["lda"]["dominant_topics"]:
        if year:
            if year not in year_topic_data:
                year_topic_data[year] = []
            year_topic_data[year].append(topic)

    if year_topic_data:
        fig, ax = plt.subplots(figsize=(12, 7))
        years = sorted(year_topic_data.keys())
        n_topics = len(results["Gaza"]["lda"]["topics"])

        # Prepare data for stacked bar chart
        topic_by_year = {topic: [] for topic in range(1, n_topics + 1)}
        for year in years:
            year_topics = Counter(year_topic_data[year])
            for topic in range(1, n_topics + 1):
                topic_by_year[topic].append(year_topics.get(topic, 0))

        # Create stacked bars
        bottom = np.zeros(len(years))
        colors = plt.cm.Set3(np.linspace(0, 1, n_topics))

        for topic in range(1, n_topics + 1):
            ax.bar(years, topic_by_year[topic], bottom=bottom, 
                  label=f'Topic {topic}', color=colors[topic-1], edgecolor='black')
            bottom += topic_by_year[topic]

        ax.set_xlabel('Year', fontsize=12, fontweight='bold')
        ax.set_ylabel('Number of Documents', fontsize=12, fontweight='bold')
        ax.set_title(f'Dominant Topics Per Year - Gaza', fontsize=14, fontweight='bold')
        ax.legend(title='Dominant Topic', bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(f"docs/visualisation/gaza_topics_per_year.png", dpi=300, bbox_inches='tight')
        plt.close()

# COMBINED ELBOW + SILHOUETTE
if "clustering" in results["Gaza"]:
    clust_data = results["Gaza"]["clustering"]
    k_range = range(2, len(clust_data["wcss"]) + 2)

    fig, ax1 = plt.subplots(figsize=(12, 6))

    # --- Elbow Curve (WCSS) ---
    ax1.plot(k_range, clust_data["wcss"], 'bo-', linewidth=2, markersize=8, label="WCSS")
    ax1.set_xlabel("Number of clusters (k)", fontsize=12, fontweight="bold")
    ax1.set_ylabel("WCSS", fontsize=12, fontweight="bold")
    ax1.grid(True, alpha=0.3)

    # Vertical best_k line
    ax1.axvline(x=clust_data["best_k"], color='red', linestyle='--', linewidth=2)

    ax2 = ax1.twinx()
    ax2.plot(k_range, clust_data["silhouette_scores"], 'o-', 
             linewidth=2, markersize=8, color="orange", label="Silhouette Score")
    ax2.set_ylabel("Silhouette Score", fontsize=12, fontweight="bold", color="orange")
    ax2.tick_params(axis='y', labelcolor="orange")

    best_score = clust_data["silhouette_scores"][clust_data["best_k"] - 2]
    ax2.axhline(y=best_score, color='orange', linestyle='--', linewidth=1, alpha=0.5)

    plt.title("Elbow vs Silhouette Comparaison", fontsize=16, fontweight="bold")
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, fontsize=11, loc="upper right")

    plt.tight_layout()
    plt.savefig(f"docs/visualisation/gaza_clustering_combined.png", 
                dpi=300, bbox_inches='tight')
    plt.close()

# K-MEANS CLUSTERING VISUALIZATION (PCA)
if "pca_coords" in clust_data:
    fig, ax = plt.subplots(figsize=(10, 8))
    pca_coords = np.array(clust_data["pca_coords"])
    labels = np.array(clust_data["labels"])

    colors = plt.cm.Set2(np.linspace(0, 1, clust_data["best_k"]))

    for cluster_id in range(clust_data["best_k"]):
        cluster_points = pca_coords[labels == cluster_id]
        ax.scatter(cluster_points[:, 0], cluster_points[:, 1], 
                  c=[colors[cluster_id]], label=f'Cluster {cluster_id}',
                  s=100, alpha=0.6, edgecolors='black', linewidth=1)

    ax.set_xlabel('PC1', fontsize=12, fontweight='bold')
    ax.set_ylabel('PC2', fontsize=12, fontweight='bold')
    ax.set_title(f'K-Means Clustering (k={clust_data["best_k"]}) - Gaza', 
                fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"docs/visualisation/gaza_kmeans_clusters.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # PCA 3 
    pca3 = PCA(n_components=3, random_state=42)
    X_pca3 = clust_data["pca_coords_3"]
    fig = plt.figure(figsize=(8,6))
    ax = fig.add_subplot(111, projection='3d')

    scatter_all = ax.scatter(
        X_pca3[:,0], X_pca3[:,1], X_pca3[:,2],
        c=labels,
        cmap='plasma',
        s=40,
        alpha=0.9
    )
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    ax.set_zlabel("PC 3")
    ax.set_title(f"TF-IDF clustered by K-Means (k={best_k}) — 3D PCA")
    plt.tight_layout()
    fig.canvas.draw()
    plt.savefig(f"docs/visualisation/gaza_kmeans_clusters_pca3.png", dpi=300, bbox_inches='tight')
    plt.close()
    

print(f"\n✅ All visualizations saved in: docs/visualisation/")


🌍 Scraping for Gaza...

📄 Reading page 1 -> Gaza...
✨ 8 link(s) found on page 1.
📄 Reading page 2 -> Gaza...
✨ 11 link(s) found on page 2.
📄 Reading page 3 -> Gaza...
✨ 7 link(s) found on page 3.
📄 Reading page 4 -> Gaza...
✨ 1 link(s) found on page 4.
📄 Reading page 5 -> Gaza...
❌ No links found — probably end of pagination.

🌍 Gaza: 27/27 code(s) extracted from link(s) successfully, downloading...
✅ PDF downloaded: A_HRC_61_70.pdf
⚠️ Unable to download https://documents.un.org/api/symbol/access?s=A/HRC/61/26&l=en&t=pdf: 404 Client Error: Not Found for url: https://documents.un.org/error
✅ PDF downloaded: A_80_399.pdf
✅ PDF downloaded: A_HRC_58_73.pdf
✅ PDF downloaded: A_HRC_58_28.pdf
✅ PDF downloaded: A_79_347.pdf
✅ PDF downloaded: A_HRC_55_28.pdf
✅ PDF downloaded: A_HRC_55_72.pdf
✅ PDF downloaded: A_78_554.pdf
✅ PDF downloaded: A_78_502.pdf
✅ PDF downloaded: A_HRC_52_76.pdf
✅ PDF downloaded: A_HRC_52_75.pdf
✅ PDF downloaded: A_77_493.pdf
✅ PDF downloaded: A_HRC_49_83.pdf
✅ PDF down